In [ ]:
'''Kaggriculture | Smart Agent Engine — Reactive Shop Router

This research submission packages the exact 0909 shop-pair replay router. It
selects from thirteen complete tapes after the first two shops are visible,
repairs weed-blocked work within the same day, advances only eligible planned
sales, and liquidates the final reachable stock. The Apache-2.0 license for the
credited reactive-sale component is included in the submission archive. Run the
next cell to create submission.tar.gz for Kaggle.
'''


In [ ]:
'''Materialize the exact multi-file Reactive Shop Router.

The controller needs its action tapes and license alongside main.py. The package
therefore keeps all three files rather than forcing this router into one file.
'''
import base64
import gzip
import hashlib
import io
import tarfile
from pathlib import Path

MAIN_SOURCE = '"""Shop plans with small, observation-based repairs. Python standard library only.\n\nThe 13 complete action tapes live in actions.json. This file contains every rule:\nchoose a plan after two shops, delay weed-blocked work within the current day,\nbring some planned sales forward one turn, and liquidate on the final turn.\n\nSell timing and shed projection follow aurax7\'s public Reactive Router:\nhttps://www.kaggle.com/code/aurax7/kaggriculture-reactive-router\nThe shop-pair routes and worker-local, same-day queues were developed here.\n"""\n\nimport copy\nimport json\nfrom collections import deque\nfrom pathlib import Path\n\nTURNS_PER_DAY = 24\nROUTE_STEP = 144\nFINAL_PLAN_STEP = 648\nLAST_STEP = 718\nSHED_CAPACITY = 100\nMAX_ORDERS = 10\nPRODUCTS = (\n    "WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON",\n    "EGG", "MILK", "WOOL", "FERTILIZER",\n)\nWEED_BLOCKED_WORK = {"PLANT", "BUILD_COOP", "BUILD_PASTURE"}\nANIMALS = {"GOOSE", "COW", "SHEEP"}\n\n# Keys are the first two shops in their observed order; values index actions.json.\n# All other pairs keep plan 0. Plan 1 is the previous yarn-market continuation.\n# Plans 3..12 are the ten distinct continuations selected in the latest search.\nSHOP_PLANS = {\n    ("BAKERY", "YARN_STORE"): 3,\n    ("BRUNCH_SPOT", "YARN_STORE"): 4,\n    ("FARMERS_MARKET", "YARN_STORE"): 5,\n    ("ICE_CREAM_SHOP", "YARN_STORE"): 6,\n    ("PET_CAFE", "YARN_STORE"): 5,\n    ("PIZZA_SHOP", "YARN_STORE"): 7,\n    ("SMOOTHIE_SHOP", "YARN_STORE"): 8,\n    ("YARN_STORE", "BAKERY"): 9,\n    ("YARN_STORE", "BRUNCH_SPOT"): 9,\n    ("YARN_STORE", "FARMERS_MARKET"): 1,\n    ("YARN_STORE", "ICE_CREAM_SHOP"): 9,\n    ("YARN_STORE", "PET_CAFE"): 10,\n    ("YARN_STORE", "PIZZA_SHOP"): 6,\n    ("YARN_STORE", "SMOOTHIE_SHOP"): 11,\n    ("YARN_STORE", "YARN_STORE"): 12,\n}\n\n\nclass FarmView:\n    """Only the current own farm, private inventory, and public prices."""\n\n    def __init__(self, observation):\n        farm = observation["farms"][observation["player"]]\n        private = observation["private"]\n        self.tiles = farm["tiles"]\n        self.positions = [farm["farmer"], *farm["hands"]]\n        self.inventories = private["inventories"]\n        self.shed = {item: max(0, int(qty)) for item, qty in private["shed"].items()}\n        self.prices = observation["market"]["prices"]\n\n    def inventory(self, worker):\n        return self.inventories[worker] if worker < len(self.inventories) else {}\n\n    def beside_shed(self, position):\n        center = len(self.tiles) // 2\n        return position[0] in (center - 1, center) and position[1] in (center - 1, center)\n\n\nclass DayState:\n    """Per-player memory; queues expire at dawn and sales expire next turn."""\n\n    def __init__(self):\n        self.plan = 0\n        self.last_step = -1\n        self.day = -1\n        self.queues = {}\n        self.sale_due_step = -1\n        self.advanced_sales = {}\n\n\ndef repair_weeds(action, view, state, step):\n    """Insert DIG without consuming the blocked action; shift only this worker."""\n    day = step // TURNS_PER_DAY\n    if day != state.day:\n        state.day = day\n        state.queues.clear()  # Unfinished work never spills into tomorrow.\n\n    workers = [action.get("farmer") or ["PASS"], *(action.get("hands") or [])]\n    for worker in range(min(len(workers), len(view.positions))):\n        queue = state.queues.setdefault(worker, deque())\n        queue.append(list(workers[worker]))\n        x, y = view.positions[worker]\n        tile = view.tiles[y][x]\n        blocked = (queue[0][0] in WEED_BLOCKED_WORK\n                   and isinstance(tile, dict) and tile.get("kind") == "WEED")\n        workers[worker] = ["DIG"] if blocked else queue.popleft()\n    action["farmer"], action["hands"] = workers[0], workers[1:]\n\n\ndef projected_shed(action, view):\n    """Estimate stock after this turn\'s nearby PICKUP, DROP and PLACE actions.\n\n    Preserve worker and inventory order: limited shed capacity can make it matter.\n    This is the qualified lightweight estimate, not a full game simulation.\n    """\n    stock = {item: view.shed.get(item, 0) for item in PRODUCTS}\n    stock.update(view.shed)\n    total = sum(stock.values())\n    workers = [action.get("farmer") or ["PASS"], *(action.get("hands") or [])]\n    for worker in range(min(len(workers), len(view.positions))):\n        if not view.beside_shed(view.positions[worker]):\n            continue\n        work = workers[worker]\n        operation = work[0] if work else "PASS"\n        inventory = view.inventory(worker)\n        if operation == "PICKUP" and len(work) >= 2 and work[1] in stock:\n            quantity = max(0, int(work[2]) if len(work) >= 3 else 1)\n            taken = min(stock[work[1]], quantity)\n            stock[work[1]] -= taken\n            total -= taken\n        elif operation == "DROP":\n            for item, held in inventory.items():\n                added = min(max(0, int(held)), max(0, SHED_CAPACITY - total))\n                if added > 0:\n                    stock[item] = stock.get(item, 0) + added\n                    total += added\n        elif operation == "PLACE" and len(work) >= 2 and work[1] not in ANIMALS:\n            item = work[1]\n            quantity = max(0, int(work[2]) if len(work) >= 3 else 1)\n            added = min(quantity, max(0, int(inventory.get(item, 0))),\n                        max(0, SHED_CAPACITY - total))\n            if added > 0:\n                stock[item] = stock.get(item, 0) + added\n                total += added\n    return stock\n\n\ndef subtract_advanced_sales(action, state, step):\n    """Remove quantities already requested one turn early, retaining order slots."""\n    if state.sale_due_step == step:\n        remaining = dict(state.advanced_sales)\n        for order in action["market"]:\n            if order and order[0] == "SELL" and len(order) >= 3:\n                item = order[1]\n                removed = min(max(0, int(order[2])), remaining.get(item, 0))\n                if removed > 0:\n                    order[2] = int(order[2]) - removed\n                    remaining[item] -= removed\n    state.advanced_sales = {}\n    state.sale_due_step = -1\n\n\ndef advance_sales(action, view, state, tape, step):\n    """Bring eligible sales from our next planned action forward by one turn."""\n    next_step = step + 1\n    if next_step > LAST_STEP or next_step % 72 == 0 or step % 4 == 0:\n        return\n    planned = {}\n    for order in tape[next_step].get("market") or []:\n        if order and order[0] == "SELL" and len(order) >= 3 and order[1] in PRODUCTS:\n            item = order[1]\n            planned[item] = planned.get(item, 0) + max(0, int(order[2]))\n    already_selling = {order[1] for order in action["market"]\n                       if order and order[0] == "SELL" and len(order) > 1}\n    stock = projected_shed(action, view)\n    for item in PRODUCTS:\n        if item in ("WHEAT", "FERTILIZER") or item in already_selling:\n            continue\n        quantity = min(stock.get(item, 0), planned.get(item, 0))\n        if quantity <= 0 or int(view.prices.get(item, 0)) < 2:\n            continue\n        if len(action["market"]) >= MAX_ORDERS:\n            break\n        action["market"].append(["SELL", item, quantity])\n        state.advanced_sales[item] = quantity\n    if state.advanced_sales:\n        state.sale_due_step = next_step\n\n\ndef liquidate(view):\n    """On the last turn, drop reachable inventory and sell the projected shed."""\n    workers = [["DROP"] if view.beside_shed(pos) and view.inventory(worker) else ["PASS"]\n               for worker, pos in enumerate(view.positions)]\n    action = {"farmer": workers[0], "hands": workers[1:], "market": []}\n    stock = projected_shed(action, view)\n    action["market"] = [["SELL", item, stock[item]] for item in PRODUCTS if stock[item] > 0]\n    action["market"].sort(key=lambda order: -int(view.prices.get(order[1], 0)) * order[2])\n    return action\n\n\nclass Policy:\n    def __init__(self, folder):\n        self.tapes = json.loads((Path(folder) / "actions.json").read_text())\n        if len(self.tapes) != 13 or any(len(tape) != LAST_STEP + 1 for tape in self.tapes):\n            raise ValueError("Expected 13 complete, 719-turn action tapes")\n        self.players = {}\n\n    def act(self, observation):\n        step, player = int(observation["step"]), int(observation["player"])\n        state = self.players.get(player)\n        if state is None or step <= state.last_step:\n            state = self.players[player] = DayState()\n        state.last_step = step\n\n        if step == ROUTE_STEP:\n            shops = observation["town"]["unlocked_shops"]\n            state.plan = SHOP_PLANS.get(tuple(shops[:2]), 0)\n        if step == FINAL_PLAN_STEP:\n            state.plan = 2\n\n        view = FarmView(observation)\n        tape = self.tapes[state.plan]\n        action = copy.deepcopy(tape[step])\n        repair_weeds(action, view, state, step)\n        subtract_advanced_sales(action, state, step)\n        advance_sales(action, view, state, tape, step)\n        action["market"] = action["market"][:MAX_ORDERS]\n        return liquidate(view) if step == LAST_STEP else action\n\n\n_POLICY = None\n\n\ndef agent(observation, configuration=None):\n    global _POLICY\n    if _POLICY is None:\n        # Kaggle\'s source loader omits __file__, but retains the code filename.\n        folder = Path(agent.__code__.co_filename).resolve().parent\n        _POLICY = Policy(folder)\n    return _POLICY.act(observation)\n'
ACTIONS_GZIP_B64 = '...'
LICENSE_TEXT = '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n'
EXPECTED_SHA256 = {'main.py': 'd6d74997dc5b483db63d8e39cafa1afeec0f366824e75107e109123f111e866b', 'actions.json': '17d503f2fd20d59f9c0f14024d1e74a8add8bb9b5561d4d908b45deecb5495ef', 'LICENSE.txt': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30'}

'''Write source assets and verify their exact content hashes.'''
assets = {
    "main.py": MAIN_SOURCE.encode(),
    "actions.json": gzip.decompress(base64.b64decode(ACTIONS_GZIP_B64)),
    "LICENSE.txt": LICENSE_TEXT.encode(),
}
for name, payload in assets.items():
    assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256[name]
    Path(name).write_bytes(payload)
compile(MAIN_SOURCE, "main.py", "exec")

'''Package the complete submission artifact expected by Kaggriculture.'''
archive = Path("submission.tar.gz")
with tarfile.open(archive, "w:gz") as tar:
    for name in ("main.py", "actions.json", "LICENSE.txt"):
        payload = assets[name]
        info = tarfile.TarInfo(name)
        info.size = len(payload)
        info.mode = 0o644
        info.mtime = 0
        tar.addfile(info, io.BytesIO(payload))
with tarfile.open(archive, "r:gz") as tar:
    assert tar.getnames() == ["main.py", "actions.json", "LICENSE.txt"]
    for name, payload in assets.items():
        assert tar.extractfile(name).read() == payload

print("variant: smart_agent_engine_reactive_shop_router_exact")
print("archive SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest())
print("submission.tar.gz: PASS")
